**Выгрузка численности населения с сайта Росстата Иркутской области**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

In [3]:
pip install requests beautifulsoup4 pandas openpyxl lxml

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import BytesIO
import re

In [5]:
# URL страницы
url = 'https://38.rosstat.gov.ru/folder/167937'

try:
    # Получаем содержимое страницы
    response = requests.get(url, timeout=30, verify=False)  # без сертификата
    response.raise_for_status()
    
    # Парсим HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Ищем ссылку на файл с нужным названием
    target_file = None
    for link in soup.find_all('a', href=True):

        link_text = link.get_text(strip=True)
        href = link['href']
        
        # Проверяем, содержит ли текст ссылки нужное название
        file_name = 'Численность мужчин и женщин'
        
        if file_name in href:
            # Формируем полный URL
            if href.startswith('http'):
                target_file = href
            elif href.startswith('/'):
                # Если ссылка относительная, добавляем домен
                base_url = 'https://38.rosstat.gov.ru'
                target_file = base_url + href
            else:
                # Если ссылка относительно текущей страницы
                target_file = url.rstrip('/') + '/' + href
            break
    
    if not target_file:
        raise FileNotFoundError("Файл с указанным названием не найден на странице")
    
    print(f"Найден файл: {target_file}")
    
    # Загружаем Excel‑файл
    file_response = requests.get(target_file, timeout=30, verify=False) # без сертификата
    file_response.raise_for_status()
    
    # Создаём объект BytesIO из содержимого файла
    excel_file = BytesIO(file_response.content)
    
    # Читаем Excel в датасет
    df = pd.read_excel(excel_file)
    
    print("Файл успешно загружен в датасет!")
#    print(f"Размер датасета: {df.shape}")
#    print("\nПервые 5 строк:")
#    print(df.head())
    
except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке страницы или файла: {e}")
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"Ошибка при обработке файла: {e}")

C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Найден файл: https://38.rosstat.gov.ru/storage/mediabank/Численность мужчин и женщин_2025.xls


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Файл успешно загружен в датасет!


In [6]:
df

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,Численность мужчин и женщин Иркутской области\...,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,оды,"Все население,",в том числе:,NaN,"Доля в общей численности населения, %",NaN
3,NaN,человек,мужчины,женщины,мужчин,женщин
4,1995,2748073,1317112,1430961,"47,9","52,1"
5,2000,2644022,1249878,1394144,"47,3","52,7"
6,2005,2524080,1175837,1348243,"46,6","53,4"
7,2006,2492143,1158099,1334044,"46,5","53,5"
8,2007,2467383,1145168,1322215,"46,4","53,6"
9,2008,2455410,1139449,1315961,"46,4","53,6"


In [7]:
path = 'E:/IrkutskProject/Data/'

#df_population = df.iloc[17:27]
columns = [
    'Год',
    'Всего',
    'Всего_среднее',
    'Мужчины',
    'Мужчины_среднее',
    'Женщины',
    'Женщины_среднее'
]

df_population = pd.DataFrame(columns=columns)
data = []

for row_number in range(df.shape[0]):
    if pd.notnull(df.iloc[row_number,0]):
        year = str(df.iloc[row_number,0])
        if '*' in year:
            year = year.replace('*','')
        if year.isdigit():
            year = int(year)
            if year >= 2016:
                new_row = []
                new_row.append(year)
                for col_number in range(1, 4): #df.shape[1]):
                    new_row.append(df.iloc[row_number,col_number])
                    if pd.notnull(df.iloc[row_number + 1,col_number]):
                         new_row.append(int((df.iloc[row_number,col_number] + df.iloc[row_number + 1,col_number])/2))
                    else:
                        new_row.append(df.iloc[row_number,col_number])
                data.append(new_row)

df_population = pd.DataFrame(data, columns=columns)
display(df_population)

,Год,Всего,Всего_среднее,Мужчины,Мужчины_среднее,Женщины,Женщины_среднее
0,2016,2415690,2414024,1112547,1111367,1303143,1302657
1,2017,2412359,2410290,1110187,1108556,1302172,1301733
2,2018,2408221,2405289,1106926,1105026,1301295,1300263
3,2019,2402358,2399358,1103127,1101971,1299231,1297387
4,2020,2396358,2388558,1100815,1096570,1295543,1291988
5,2021,2380759,2372103,1092325,1089220,1288434,1282882
6,2022,2363447,2353903,1086116,1081052,1277331,1272851
7,2023,2344360,2337448,1075988,1072490,1268372,1264958
8,2024,2330537,2326414,1068992,1067116,1261545,1259298
9,2025,2322292,2322292,1065241,1065241,1257051,1257051


In [8]:
df_population.to_csv(f'{path}Common/population_irkutsk.csv', index=False)